# Eksplorasi Data & Uji Heterogenitas Lintas Platform

Notebook ini tidak hanya melakukan EDA standar, tetapi secara spesifik mengukur **Domain Shift (Heterogenitas)** antara korpus ulasan Tokopedia dan Shopee menggunakan metrik **Out-Of-Vocabulary (OOV) Rate** dan **Jensen-Shannon Divergence (JSD)**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from collections import Counter
from scipy.spatial import distance
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Memuat Dataset Eksperimen
Memuat data mentah yang telah digabung (`merged_labeled.csv`) sebelum dipecah ke dalam berbagai kondisi eksperimen.

In [ ]:
DATA_PATH = "../data/raw/merged/multi_platform/merged_labeled.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Total Data Master: {len(df)} baris")
    display(df.head())
else:
    print("File merged_labeled.csv tidak ditemukan. Jalankan merger.py terlebih dahulu.")

### 2. Analisis Distribusi Panjang Teks (Token Count)

In [ ]:
if 'df' in locals():
    df['token_count'] = df['review_text'].astype(str).apply(lambda x: len(x.split()))
    
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x='source_platform', y='token_count', palette='Set2', hue='source_platform')
    plt.title('Distribusi Panjang Teks: Tokopedia vs Shopee')
    plt.ylim(0, 80) # Dibatasi agar outlier ekstrem tidak merusak visualisasi
    plt.show()
    
    display(df.groupby('source_platform')['token_count'].describe())

### 3. Uji Heterogenitas 1: OOV (Out-Of-Vocabulary) Rate
Kita akan menggunakan modul `AutoTokenizer` dari IndoBERT. Jika sebuah kata sering dipecah menjadi sub-token (WordPiece) yang banyak, atau digantikan dengan `[UNK]`, ini mengindikasikan bahwa kata tersebut adalah *slang/typo* yang tidak dikenali (OOV) oleh representasi internal IndoBERT.

In [ ]:
if 'df' in locals():
    from transformers import AutoTokenizer
    
    # Load IndoBERT Tokenizer
    try:
        model_dir = "../model/indobert_raw"
        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        
        def calculate_fragmentation_ratio(text):
            words = str(text).split()
            if not words:
                return 0
            tokens = tokenizer.tokenize(str(text))
            # Jika jumlah token jauh melebihi jumlah kata, berarti banyak kata terfragmentasi (OOV/Slang)
            return len(tokens) / len(words)
        
        print("Menghitung rasio fragmentasi sub-word (OOV proxy)...")
        # Ambil sampel 5000 agar komputasi tidak terlalu lama di EDA
        df_sample = df.groupby('source_platform').sample(n=min(5000, len(df)//2), random_state=42)
        df_sample['fragmentation_ratio'] = df_sample['review_text'].apply(calculate_fragmentation_ratio)
        
        plt.figure(figsize=(8, 5))
        sns.kdeplot(data=df_sample, x='fragmentation_ratio', hue='source_platform', fill=True, common_norm=False)
        plt.axvline(1.0, color='r', linestyle='--', label='1 Kata = 1 Token (Ideal Formal)')
        plt.title('Kepadatan Fragmentasi Token IndoBERT (Indikasi Tingkat Slang)')
        plt.xlabel('Rasio Sub-Word per Kata (>1 berarti banyak OOV)')
        plt.legend()
        plt.show()
        
        print("Rata-rata Fragmentasi (Semakin tinggi, semakin jauh dari bahasa formal):")
        display(df_sample.groupby('source_platform')['fragmentation_ratio'].mean())
        
    except Exception as e:
        print(f"Tokenizer gagal dimuat. Pastikan folder model/indobert_raw tersedia. Error: {e}")

### 4. Uji Heterogenitas 2: Jensen-Shannon Divergence (JSD)
Mengukur seberapa jauh perbedaan distribusi frekuensi kata antara korpus Tokopedia dan Shopee. JSD bernilai 0 jika distribusi kata identik, dan mendekati 1 jika sama sekali tidak ada kata yang tumpang tindih.

In [ ]:
if 'df' in locals():
    def get_word_distribution(texts, top_k=5000):
        all_words = " ".join(texts).lower().split()
        counter = Counter(all_words)
        # Ambil top K kata
        common = counter.most_common(top_k)
        words = [w[0] for w in common]
        counts = np.array([w[1] for w in common], dtype=np.float64)
        # Normalize menjadi probabilitas (PDF)
        probs = counts / counts.sum()
        return dict(zip(words, probs))

    # Pisahkan teks per platform
    toped_texts = df[df['source_platform'] == 'tokopedia']['review_text'].astype(str).tolist()
    shopee_texts = df[df['source_platform'] == 'shopee']['review_text'].astype(str).tolist()
    
    if toped_texts and shopee_texts:
        dist_toped = get_word_distribution(toped_texts)
        dist_shopee = get_word_distribution(shopee_texts)
        
        # Gabungkan vocabulary dari kedua platform untuk membandingkan probabilitas
        unified_vocab = list(set(list(dist_toped.keys()) + list(dist_shopee.keys())))
        
        p_toped = np.array([dist_toped.get(w, 1e-10) for w in unified_vocab])
        p_shopee = np.array([dist_shopee.get(w, 1e-10) for w in unified_vocab])
        
        # Normalisasi ulang karena ada penambahan smoothing 1e-10
        p_toped /= p_toped.sum()
        p_shopee /= p_shopee.sum()
        
        jsd_score = distance.jensenshannon(p_toped, p_shopee) ** 2
        
        print(f"========= HASIL UJI DISTRIBUSI KATA =========")
        print(f"Jensen-Shannon Divergence (JSD) Score: {jsd_score:.4f}")
        
        if jsd_score > 0.05:
            print("[KESIMPULAN]: Terdapat Domain Shift (Heterogenitas) yang signifikan antar platform.")
            print("Perbedaan gaya bahasa/slang membenarkan kebutuhan skenario Cross-Platform (LOSO).")
        else:
            print("[KESIMPULAN]: Distribusi kata identik/homogen. Domain shift minimal.")
    else:
        print("Data dari salah satu platform tidak mencukupi.")